email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "google_genai:gemini-3.1-flash-lite",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'The inbox contains an email from Jane (jane@example.com) asking to grab coffee next week. Would you like me to draft a response for you?', 'extras': {'signature': 'EoUBCoIBAWkUfROXwfKgzLBbOBWPP7s2pN/KzNk4QSrFm3bnVcFaXYAifyODo8cFUgetSGUoq32bJjPK8OTzoI5+hge6NvilG87NyKDZx/Wb0SyOGZrX58rd0BQrvCLOu39KurDDJnVMMohZCGi3GzqVKiC0LFnulot6YWwhksGOtgYQ8/1D/w=='}}]


In [9]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

[]


In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane, 

That sounds great! I'd love to grab coffee while you're in town next week. Let me know what day and time works best for you.

Best,
Julie


In [11]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'The email has been sent to Jane.', 'extras': {'signature': 'EnMKcQFpFH0TSAuEh5sVhHy29wfGoX54iD0ugCt5kErF/4sQmsSOc1CbadZ5L+4mPOl+2Kfc2I6e4+/wwWQLp9xlaTmq01ogFAcfzXdnmUAIm8VsQ9WpHpFXcF73RM2DnKFLhernsDr+LqB30NJDLc6G+CXB'}}]


In [12]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='cb20a88e-9da6-4f44-ad15-e9d4343b74dc'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'authenticate', 'arguments': '{"email": "julie@example.com", "password": "password123"}'}, '__gemini_function_call_thought_signatures__': {'call_16617987616071728030': 'EnMKcQFpFH0T2S9zqL5dUfXB5U/UcGGNZHuOC9OZWhTPZNXHrWwtcu184WsC78U+3SeCUqVuSdrELl/T12gmY14rGynMI5g2oAam43cfOomLIYj1OkYL9z4YCIy//MPU4fzxz6UhFbLVqZ57RiQiAciZogHl'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1d004-696c-7d22-9168-3bbf0ba03a95-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_16617987616071728030', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens'